# Sessao 3 - OpenAI compat + Memory + Copa do Mundo

Este notebook faz 4 coisas, em ordem simples:
1. Cria conversas com `memory_subject_id` para testar memoria entre conversas
2. Semeia contexto de Copa do Mundo e valida short-term + long-term
3. Mantem o codigo enxuto, sem blocos de excecao desnecessarios
4. No final, usa DuckDuckGo como tool da LLM via OpenAI-compatible


In [1]:
!pip install openai python-dotenv httpx oci_genai_auth ddgs



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1) Carregar variaveis de ambiente e escolher autenticacao por regiao

`load_dotenv()` continua ativo. A regra fica assim:
- Se `OCI_REGION=sa-saopaulo-1`: usa IAM com arquivo `config` (User Principal)
- Se `OCI_REGION=us-chicago-1`: usa API key
- Outras regioes: usa API key por padrao


In [2]:
import os
import json
import time
import httpx
from ddgs import DDGS
from dotenv import load_dotenv
from openai import OpenAI
from oci_genai_auth import OciUserPrincipalAuth

load_dotenv(override=True)

OCI_REGION = os.getenv("OCI_REGION", "")
OCI_OPENAI_API_KEY = os.getenv("OCI_OPENAI_API_KEY", "")
PROJECT_ID = os.getenv("PROJECT_ID", "").split("#", 1)[0].strip()
MODEL_ID = os.getenv("MODEL_ID", "xai.grok-4-fast-non-reasoning")
OCI_PROFILE = os.getenv("OCI_PROFILE", "DEFAULT")
OCI_CONFIG_FILE = os.getenv("OCI_CONFIG_FILE", "./config")
MEMORY_SUBJECT_ID = os.getenv("MEMORY_SUBJECT_ID", "copa-user-001")

BASE_URL = f"https://inference.generativeai.{OCI_REGION}.oci.oraclecloud.com/openai/v1"


client = OpenAI(
    api_key=OCI_OPENAI_API_KEY,
    base_url=BASE_URL,
    project=PROJECT_ID,
)


print("BASE_URL:", BASE_URL)
print("OCI_REGION:", OCI_REGION)
print("MODEL_ID:", MODEL_ID)

BASE_URL: https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/openai/v1
OCI_REGION: us-chicago-1
MODEL_ID: xai.grok-4-fast-non-reasoning


## 2) Criar conversa A para armazenar memoria

Criamos uma conversa com `memory_subject_id` e politica `store_only`,
depois conferimos o metadata salvo no servidor.


In [3]:
conv_a_metadata = {
    "memory_subject_id": MEMORY_SUBJECT_ID,
    "memory_access_policy": "store_only",
}

conv_a = client.conversations.create(metadata=conv_a_metadata)
conv_a_id = conv_a.id
conv_a_check = client.conversations.retrieve(conv_a_id)
conv_a_check_dict = conv_a_check.to_dict() if hasattr(conv_a_check, "to_dict") else json.loads(conv_a_check.model_dump_json())

print(json.dumps({
    "conversation_id": conv_a_id,
    "requested_metadata": conv_a_metadata,
    "stored_metadata": conv_a_check_dict.get("metadata", {}),
}, indent=2, ensure_ascii=False))


{
  "conversation_id": "conv_ord_mgrs27en8z78iaysoon0bir2wgfha5nn5bl3pw0cetzvs8jf",
  "requested_metadata": {
    "memory_subject_id": "copa-user-001",
    "memory_access_policy": "store_only"
  },
  "stored_metadata": {
    "memory_subject_id": "copa-user-001",
    "memory_access_policy": "store_only",
    "short_term_memory_optimization": "true"
  }
}


## 3) Semear fatos da Copa na conversa A

Guardamos 3 fatos de contexto. Isso alimenta short-term (mesma conversa)
e long-term (mesmo `memory_subject_id`).


In [4]:
copa_facts = [
    "Na fase de grupos da Copa, vitoria vale 3 pontos e empate vale 1.",
    "Em mata-mata, se empatar no tempo normal, pode haver prorrogacao e penaltis.",
    "Criterios comuns de desempate incluem saldo de gols e gols marcados.",
]

for fact in copa_facts:
    ack = client.responses.create(
        model=MODEL_ID,
        conversation=conv_a_id,
        input=f"Memorize este contexto para uso futuro do mesmo usuario: {fact}",
    )
    print("FACT:", fact)
    print("ACK:", getattr(ack, "output_text", "")[:180], "\n")


FACT: Na fase de grupos da Copa, vitoria vale 3 pontos e empate vale 1.
ACK: Entendi! Memorizei o contexto sobre a fase de grupos da Copa: vitória vale 3 pontos, empate vale 1. Posso usá-lo em interações futuras com você. O que mais precisa? 

FACT: Em mata-mata, se empatar no tempo normal, pode haver prorrogacao e penaltis.
ACK: Entendi! Memorizei o contexto sobre mata-mata: se empatar no tempo normal, pode haver prorrogação e pênaltis. Vou considerar isso em interações futuras com você. Algo mais? 

FACT: Criterios comuns de desempate incluem saldo de gols e gols marcados.
ACK: Entendi! Memorizei o contexto sobre critérios de desempate: incluem saldo de gols e gols marcados. Vou aplicar isso em interações futuras com você. Precisa de mais alguma coisa? 



## 4) Validar short-term (mesma conversa A)

Perguntamos na mesma conversa. Se vier os fatos, short-term esta funcionando.


In [5]:
short_query = "Liste objetivamente as regras da Copa que eu acabei de te passar."

short_resp = client.responses.create(
    model=MODEL_ID,
    conversation=conv_a_id,
    input=short_query,
)

short_answer = getattr(short_resp, "output_text", "")

print(json.dumps({
    "conversation_id": conv_a_id,
    "query": short_query,
    "answer": short_answer,
}, indent=2, ensure_ascii=False))


{
  "conversation_id": "conv_ord_mgrs27en8z78iaysoon0bir2wgfha5nn5bl3pw0cetzvs8jf",
  "query": "Liste objetivamente as regras da Copa que eu acabei de te passar.",
  "answer": "Aqui está uma lista objetiva das regras da Copa que você me passou para memorizar:\n\n- **Fase de grupos**: Vitória vale 3 pontos; empate vale 1 ponto.\n- **Mata-mata**: Se empatar no tempo normal, pode haver prorrogação e pênaltis.\n- **Critérios de desempate**: Incluem saldo de gols e gols marcados."
}


## 5) Validar long-term (conversa B nova, mesmo subject)

Criamos nova conversa com `recall_only`, aguardamos alguns segundos
para ingestao de memoria e validamos a recuperacao.


In [6]:
conv_b_metadata = {
    "memory_subject_id": MEMORY_SUBJECT_ID,
    "memory_access_policy": "recall_only",
}

time.sleep(8)

conv_b = client.conversations.create(metadata=conv_b_metadata)
conv_b_id = conv_b.id
conv_b_check = client.conversations.retrieve(conv_b_id)
conv_b_check_dict = conv_b_check.to_dict() if hasattr(conv_b_check, "to_dict") else json.loads(conv_b_check.model_dump_json())

long_query = "Quais informacoes voce lembra sobre regras da Copa que eu compartilhei antes?"
long_resp = client.responses.create(
    model=MODEL_ID,
    conversation=conv_b_id,
    input=long_query,
)
long_answer = getattr(long_resp, "output_text", "")

hit_count = 0
for fact in copa_facts:
    if fact.lower().replace(".", "") in long_answer.lower():
        hit_count += 1

print(json.dumps({
    "conversation_id": conv_b_id,
    "conversation_metadata": conv_b_check_dict.get("metadata", {}),
    "query": long_query,
    "answer": long_answer,
    "matched_facts": hit_count,
}, indent=2, ensure_ascii=False))


{
  "conversation_id": "conv_ord_ui43lctr28hadm36afjqgkrfnjks9bp46kpjaby42683lr4j",
  "conversation_metadata": {
    "memory_subject_id": "copa-user-001",
    "memory_access_policy": "recall_only",
    "short_term_memory_optimization": "true"
  },
  "query": "Quais informacoes voce lembra sobre regras da Copa que eu compartilhei antes?",
  "answer": "Com base no que você compartilhou em conversas anteriores, aqui está um resumo objetivo das regras da Copa que você mencionou (provavelmente se referindo a um torneio de futebol, como a Copa do Mundo ou similar). Eu não tenho memória permanente entre sessões, mas recuperei isso de interações passadas:\n\n- **Fase de grupos**: \n  - Vitória vale 3 pontos.\n  - Empate vale 1 ponto.\n  - Derrota vale 0 pontos.\n\n- **Fase mata-mata**: \n  - Em caso de empate no tempo normal, há prorrogação (30 minutos extras).\n  - Se persistir o empate, a partida é decidida nos pênaltis.\n\n- **Critérios de desempate** (para classificação na fase de grupos o

## 6) DDG como tool da LLM (OpenAI-compatible)

Agora a propria LLM decide quando chamar a tool `ddg_search`.
Implementamos a function local e fechamos o loop de tool calling com `responses.create`.


In [7]:
def ddg_search(query: str, max_results: int = 5):
    with DDGS() as ddgs:
        items = list(ddgs.text(query, max_results=max_results))

    results = []
    for item in items:
        results.append({
            "title": item.get("title", ""),
            "url": item.get("href", ""),
            "snippet": item.get("body", ""),
        })

    return {"query": query, "results": results}


ddg_tool = {
    "type": "function",
    "name": "ddg_search",
    "description": "Busca na web com DuckDuckGo e retorna titulo, url e resumo.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Termo de busca"},
            "max_results": {"type": "integer", "minimum": 1, "maximum": 10, "default": 5},
        },
        "required": ["query"],
    },
}

web_question = "Quais sao os criterios de desempate da fase de grupos da Copa 2026? Cite fontes."

resp = client.responses.create(
    model=MODEL_ID,
    conversation=conv_b_id,
    input=web_question,
    tools=[ddg_tool],
)

while True:
    calls = [
        item for item in resp.output
        if getattr(item, "type", "") == "function_call" and getattr(item, "name", "") == "ddg_search"
    ]

    if not calls:
        break

    tool_outputs = []
    for call in calls:
        args = json.loads(call.arguments or "{}")
        result = ddg_search(
            query=args.get("query", ""),
            max_results=int(args.get("max_results", 5)),
        )
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result, ensure_ascii=False),
        })

    resp = client.responses.create(
        model=MODEL_ID,
        conversation=conv_b_id,
        input=tool_outputs,
        tools=[ddg_tool],
    )

print("Pergunta:", web_question)
print("\nResposta final:")
print(resp.output_text)




Pergunta: Quais sao os criterios de desempate da fase de grupos da Copa 2026? Cite fontes.

Resposta final:
### Critérios de Desempate na Fase de Grupos da Copa do Mundo FIFA 2026

A Copa do Mundo FIFA 2026, sediada por Canadá, México e Estados Unidos, manterá o formato tradicional de fase de grupos com 12 grupos de 4 equipes cada (total de 48 seleções), onde as duas melhores de cada grupo avançam diretamente para as oitavas de final (total de 32 times). Os critérios de desempate para equipes empatadas em pontos seguem as regras padrão da FIFA, semelhantes às edições anteriores (como 2022), sem alterações significativas anunciadas até o momento. Eles são aplicados na seguinte ordem hierárquica, primeiro para desempates entre duas ou mais equipes:

1. **Maior número de pontos obtidos no confronto direto** (entre as equipes empatadas). Se ainda houver empate, vai para o próximo critério.
2. **Maior saldo de gols no confronto direto** (gols marcados menos gols sofridos nas partidas entre 